# Set Up pwd and auto updates

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)
os.chdir(PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

%pwd

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry

get_road_geometry()
get_expected_counts()

# Define Sweep

## Sweep space vs fixed params

`sweep_space` defines axes to vary — the cross-product of all lists becomes one config per combo.  
`fixed` defines everything else, shared by every config.  
Any param that appears in `sweep_space` must **not** also appear in `fixed`.

---

### Seeds
```python
# Run 3 independent seeds per structural config
"seed": [42, 43, 44]
```

---

### Schedules (`ScheduleSpecs`)

Used for `bus_interval_schedule`, `traffic_percentile_schedule`, `crashes_schedule`.

```python
# Static — same value every day
ScheduleSpecs("static", 30)

# List — explicit value per day (len must equal n_days)
ScheduleSpecs("list", [50, 70, 85])

# Dist — drawn from a scipy distribution each day
from scipy.stats import norm
ScheduleSpecs("dist", dist=norm(loc=50, scale=10))
```

Example in sweep_space:
```python
"traffic_percentile_schedule": [
    ScheduleSpecs("static", 50),
    ScheduleSpecs("list", [50, 70, 85]),   # requires n_days=3
]
```

---

### Bus interval (via schedule)
```python
"bus_interval_schedule": [ScheduleSpecs("static", v) for v in [15, 30, 60]]
# 0 = no bus service
"bus_interval_schedule": [ScheduleSpecs("static", 0), ScheduleSpecs("static", 30)]
```

---

### Tolling (`TollConfig`)
```python
"toll": [
    TollConfig.static(car=0.0),                  # no toll
    TollConfig.static(car=10.0),                 # flat $10
    TollConfig(                                  # PI controller
        signal=VolumeSignal(),
        transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
        update_every_n_steps=60,
        rounding=0.10,
    ),
]
```

---

### Population size (and other `PopulationParams` fields)

`population_size` and any other `PopulationParams` field can go directly in `sweep_space` — `build_sweep_configs` detects and wraps them automatically.

```python
"population_size": [500, 1000, 1500]

# Other PopulationParams fields work the same way
"prior_bus": [30.0, 50.0, 70.0]
"time_decay_rate": [0.05, 0.1, 0.2]
```

To pass a full custom `PopulationParams` object (e.g. to vary a scipy dist):
```python
from scipy.stats import lognorm
"population_params": [
    PopulationParams(population_size=1000, value_of_time=lognorm(s=0.64, scale=40/60)),
    PopulationParams(population_size=1000, value_of_time=0.5),  # scalar VoT
]
```

---

### Data collection tier (in `fixed` or `sweep_space`)
```python
# Typically fixed — set once for the whole sweep
"hybrid_collector_config": SWEEP_SUMMARIES_ONLY    # fastest, summaries only
"hybrid_collector_config": SWEEP_FULL_TIER1        # adds per-step tier1 time series
```

---

### Other common `fixed` params
```python
fixed = dict(
    n_days=3,
    start_hr=7,
    max_steps=99999,
    bus_capacity=60,
    bus_user_fee=0.0,
    road_path="data/roads/hw210_sl_and_curvs.parquet",
    ecs_path="data/vehicle_counts/expected_counts_seconds.csv",
    hybrid_collector_config=SWEEP_SUMMARIES_ONLY,
)
```

In [ ]:
from season.schedule_helpers import static_schedule, normal_schedule, realistic_schedule, plot_schedule



static_schedule(50),
normal_schedule(70, std=5)
# # 14-day schedules
# schedules = [
#     static_schedule(50),
#     static_schedule(80),
#     normal_schedule(70, std=15),
#     realistic_schedule(14, target_mean_tp=80, months=[1,2,3]),
#     realistic_schedule(14, target_mean_tp=60, months=[12,1,2,3,4]),
# ]







[
    normal_schedule(50, std=5),
    normal_schedule(60, std=5),
    normal_schedule(70, std=5),
    normal_schedule(80, std=5),
    normal_schedule(90, std=5),
    realistic_schedule(14, target_mean_tp=50, months=None),
    realistic_schedule(14, target_mean_tp=60, months=[12,1,2,3,4,5]),
    realistic_schedule(14, target_mean_tp=70, months=None),
    realistic_schedule(14, target_mean_tp=75, months=None),
    realistic_schedule(14, target_mean_tp=80, months=None)
]

In [ ]:
from traffic.model.tolling import (
    TollConfig, VolumeSignal, PITransform, PiecewiseLinearTransform, StepTransform
)

# ── Tolling configs ──────────────────────────────────────────────

# 3 Static flat tolls
toll_flat_5    = TollConfig.static(car=5.0)     # baseline: no toll
toll_flat_10 = TollConfig.static(car=10.0)    # moderate flat
toll_flat_20 = TollConfig.static(car=20.0)    # aggressive flat

# 7 Dynamic tolls (all use VolumeSignal — instantaneous vehicle count)

# PI controllers — vary aggressiveness via target and gains
toll_pi_loose = TollConfig(
    signal=VolumeSignal(),
    transform=PITransform(target=50, kp=0.3, ki=0.02, toll_min=0, toll_max=30),
    update_every_n_steps=60,
)
toll_pi_moderate = TollConfig(
    signal=VolumeSignal(),
    transform=PITransform(target=35, kp=0.5, ki=0.05, toll_min=0, toll_max=40),
    update_every_n_steps=60,
)
toll_pi_tight = TollConfig(
    signal=VolumeSignal(),
    transform=PITransform(target=25, kp=0.8, ki=0.08, toll_min=0, toll_max=50),
    update_every_n_steps=60,
)
toll_pi_aggressive = TollConfig(
    signal=VolumeSignal(),
    transform=PITransform(target=20, kp=1.0, ki=0.10, toll_min=0, toll_max=60),
    update_every_n_steps=60,
)

# Piecewise linear — toll ramps up above a vehicle threshold
toll_pwl_gentle = TollConfig(
    signal=VolumeSignal(),
    transform=PiecewiseLinearTransform(threshold=80, slope=0.10, base=5.0),
    update_every_n_steps=60,
)
toll_pwl_steep = TollConfig(
    signal=VolumeSignal(),
    transform=PiecewiseLinearTransform(threshold=60, slope=0.20, base=8.0),
    update_every_n_steps=60,
)

# Step — binary on/off toll at a congestion threshold
toll_step = TollConfig(
    signal=VolumeSignal(),
    transform=StepTransform(threshold=50, toll=15.0),
    update_every_n_steps=60,
)

# ── Bus headways ─────────────────────────────────────────────────
bus_intervals = [5, 10, 15, 30]

# ── Assemble toll list ───────────────────────────────────────────
toll_configs = {
    "no_toll":        toll_none,
    "flat_10":        toll_flat_10,
    "flat_20":        toll_flat_20,
    "pi_loose":       toll_pi_loose,
    "pi_moderate":    toll_pi_moderate,
    "pi_tight":       toll_pi_tight,
    "pi_aggressive":  toll_pi_aggressive,
    "pwl_gentle":     toll_pwl_gentle,
    "pwl_steep":      toll_pwl_steep,
    "step_50":        toll_step,
}

# Total configs: 10 schedules × 10 tolls × 4 bus intervals = 400 combos
# At 2-3 seeds each → 800-1200 seasons


In [ ]:
from season.parallel import build_sweep_configs
from season.configs import ScheduleSpecs, PopulationParams
from traffic.model.tolling import TollConfig
from traffic.model.tolling import FlowSignal, PITransform

from traffic.model.hybrid_collector import DataCollectionConfig, Tier2Config

VALIDATION_TIER2 = DataCollectionConfig(
    tier2=Tier2Config(
        sample_interval=30,
        max_samples=5000,
        max_agents_per_sample=3000,
    ),
)

sweep_space = {
    "toll": [
    TollConfig(signal=FlowSignal(), transform=PITransform(target=0.20, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.20, kp=10, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.20, kp=5, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),

        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.10, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.10, kp=10, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.10, kp=5, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),

        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.15, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.15, kp=10, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.15, kp=5, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),

        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.22, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.22, kp=10, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.22, kp=5, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),

        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.25, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.25, kp=10, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=0.25, kp=5, ki=30, toll_min=0, toll_max=100), update_every_n_steps=30)

    ],
    "seed": [42, 43],
}

fixed = dict(
    run_description="flow_test2",
    n_days=1,
    start_hr=7,
    population_size=4000,
    traffic_percentile_schedule = ScheduleSpecs("static", 87), 
    bus_interval_schedule=ScheduleSpecs("static", 15),
    crashes_schedule=ScheduleSpecs("static", 0),
   # data_collection=VALIDATION_TIER2,
)

configs = build_sweep_configs(sweep_space, fixed, base_seed=42)

print(f"{len(configs)} configs generated")


configs[0]

# Run Sweep

In [ ]:
from season.parallel import ParallelSweepRunner

runner = ParallelSweepRunner(configs, max_workers=8)
df = runner.run()
df

# Results

In [ ]:
print(f"Completed: {len(runner.results)}, Failed: {len(runner.failures)}")
print(f"Saved to: {runner.output_dir}")
